# Ising Trace Diagnostics

Run `just example ising_1d` from the repository root first. The example writes `target/ising_1d_trace.csv`, which this notebook reads with Polars for trace inspection, acceptance statistics, ACFs, and integrated autocorrelation times. Rust ACF and time estimates are also exported as `target/ising_1d_acf.csv` and `target/ising_1d_autocorrelation_time.csv`.

In Colab or another external notebook runtime, install `polars` and `matplotlib` if needed, then upload or mount the generated CSV. Set `MCMC_TRACE_PATH` to an explicit CSV path, or set `MCMC_REPO_ROOT` when the runtime does not start in the repository. Explicit configuration is authoritative and does not fall back to ambient directories. Set `MCMC_NOTEBOOK_OUTPUT_DIR` when figures must be written separately from the input data.

In [ ]:
import csv
import json
import math
import os
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

fixed_columns = ("chain_id", "step", "accepted", "proposed", "log_prob")
required_observables = {"energy", "magnetization"}
configured_trace = os.environ.get("MCMC_TRACE_PATH")
configured_root = os.environ.get("MCMC_REPO_ROOT")
runtime_root = Path.cwd().resolve()
if configured_trace:
    repo_root = runtime_root
    trace_candidates = [Path(configured_trace).expanduser()]
elif configured_root:
    repo_root = Path(configured_root).expanduser().resolve()
    repository_markers = (
        repo_root / "Cargo.toml",
        repo_root / "examples/ising_1d.rs",
    )
    if not all(marker.is_file() for marker in repository_markers):
        expected = ", ".join(str(marker) for marker in repository_markers)
        msg = f"MCMC_REPO_ROOT must identify this repository. Expected marker files: {expected}."
        raise FileNotFoundError(msg)
    trace_candidates = [repo_root / "target/ising_1d_trace.csv"]
else:
    repository_candidates = (runtime_root, runtime_root.parent)
    trace_candidates = [candidate / "target/ising_1d_trace.csv" for candidate in repository_candidates]
trace_path = next((candidate.resolve() for candidate in trace_candidates if candidate.is_file()), None)
if trace_path is None:
    checked = ", ".join(str(candidate) for candidate in trace_candidates)
    msg = f"Could not find Ising trace CSV. Checked {checked}. Run `just notebook-check` first."
    raise FileNotFoundError(msg)
if not configured_trace and not configured_root:
    repo_root = trace_path.parent.parent

with trace_path.open(encoding="utf-8", newline="") as trace_stream:
    header = next(csv.reader(trace_stream), [])
if len(header) != len(set(header)):
    msg = "Trace CSV column names must be unique."
    raise ValueError(msg)
if tuple(header[: len(fixed_columns)]) != fixed_columns:
    msg = f"Trace CSV must start with the fixed columns {fixed_columns}."
    raise ValueError(msg)
observable_columns = header[len(fixed_columns) :]
missing_observables = required_observables.difference(observable_columns)
if missing_observables:
    missing = ", ".join(sorted(missing_observables))
    msg = f"Trace CSV is missing required observables: {missing}."
    raise ValueError(msg)

schema_overrides = {
    "chain_id": pl.UInt64,
    "step": pl.UInt64,
    "accepted": pl.Boolean,
    "proposed": pl.Boolean,
    "log_prob": pl.Float64,
    **dict.fromkeys(observable_columns, pl.Float64),
}
trace = pl.read_csv(trace_path, schema_overrides=schema_overrides)
if trace.is_empty():
    msg = "Trace CSV must contain at least one row."
    raise ValueError(msg)
if sum(trace.null_count().row(0)):
    msg = "Trace CSV must not contain null values."
    raise ValueError(msg)
if (trace["accepted"] & ~trace["proposed"]).any():
    msg = "Every accepted step must have a concrete proposal."
    raise ValueError(msg)
for column in ("log_prob", *observable_columns):
    if not trace[column].is_finite().all():
        msg = f"Trace column {column!r} must contain only finite values."
        raise ValueError(msg)

key_columns = ["chain_id", "step"]
trace_keys = trace.select(key_columns)
if trace_keys.is_duplicated().any():
    msg = "Each (chain_id, step) pair must be unique."
    raise ValueError(msg)
if not trace_keys.equals(trace.sort(key_columns).select(key_columns)):
    msg = "Trace rows must be sorted by chain_id and step."
    raise ValueError(msg)
trace.head()

In [ ]:
summary = (
    trace.group_by("chain_id")
    .agg(
        pl.len().alias("steps"),
        pl.col("accepted").sum().alias("accepted"),
        pl.col("proposed").sum().alias("proposed"),
        pl.col("energy").mean().alias("mean_energy"),
        pl.col("magnetization").mean().alias("mean_magnetization"),
    )
    .with_columns(
        (pl.col("accepted") / pl.col("steps")).alias("step_acceptance_rate"),
        pl.when(pl.col("proposed") > 0).then(pl.col("accepted") / pl.col("proposed")).otherwise(None).alias("proposal_acceptance_rate"),
    )
    .sort("chain_id")
)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for chain_trace in trace.partition_by("chain_id", maintain_order=True):
    chain_id = chain_trace["chain_id"][0]
    ax.plot(chain_trace["step"], chain_trace["energy"], label=f"chain {chain_id}")
ax.set_title("Open-boundary 1-D Ising energy trace")
ax.set_xlabel("step")
ax.set_ylabel("energy")
if trace["chain_id"].n_unique() > 1:
    ax.legend()
fig.tight_layout()
configured_output_dir = os.environ.get("MCMC_NOTEBOOK_OUTPUT_DIR")
figure_output_dir = Path(configured_output_dir).expanduser().resolve() if configured_output_dir else repo_root / "target/notebooks"
figure_output_dir.mkdir(parents=True, exist_ok=True)
figure_output = figure_output_dir / "ising_energy_trace.png"
fig.savefig(
    figure_output,
    dpi=144,
    metadata={
        "Title": "Open-boundary 1-D Ising energy trace",
        "Description": f"Source={trace_path.name}; rows={trace.height}; chains={trace['chain_id'].n_unique()}",
    },
)
figure_output

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for chain_trace in trace.partition_by("chain_id", maintain_order=True):
    chain_id = chain_trace["chain_id"][0]
    ax.plot(
        chain_trace["step"],
        chain_trace["magnetization"],
        label=f"chain {chain_id}",
    )
ax.set_title("Magnetization trace")
ax.set_xlabel("step")
ax.set_ylabel("magnetization per spin")
if trace["chain_id"].n_unique() > 1:
    ax.legend()
fig.tight_layout()

The exported columns are intentionally plain: `chain_id`, `step`, `accepted`, `proposed`, `log_prob`, then one or more floating-point observables. The loader preserves extra observable columns while requiring the two plotted above. `step_acceptance_rate` matches the Rust `Trace::acceptance_rate` contract (accepted steps divided by all recorded steps); `proposal_acceptance_rate` separately conditions on steps that had a concrete proposal. The sections below add ESS, ESS-rate, and classical split R-hat.

## Autocorrelation and integrated time

Analyze each observable and chain separately, in order, with uniform sample spacing. Keep rejected steps and no-proposal self-loops. The example already discarded 5,000 burn-in steps; set `analysis_discard` below for external traces needing additional warm-up removal. This choice is the analyst's responsibility.

The notebook independently implements the Rust estimator: center by the sample mean and use the same divisor N for every autocovariance, giving rho[k] = sum((x[i]-mean)(x[i+k]-mean)) / sum((x[i]-mean)^2). Pair lags (0,1), (2,3), ...; stop before the first nonpositive pair and replace retained pair sums with cumulative minima. Then tau = -1 + 2 sum(pairs). This is [Geyer's initial monotone sequence estimator](https://www.stat.umn.edu/geyer/mcmc/library/mcmc/html/initseq.html), justified for stationary reversible chains with finite variance and summable correlations. An unpaired final lag is unused.

Independent samples have true tau = 1; anticorrelated samples can have tau below one. Lag and tau use recorded-sample intervals; multiply tau by the step spacing for transition-step units. For this 50-spin Ising example, divide transition-step time by 50 for sweeps. Missing truncation, constant/short traces, and nonpositive estimates are reported as unavailable. A found window is not a convergence or trace-length guarantee. Repeat with longer traces and larger lag budgets. The fewer-than-50-times flag below is only a heuristic caution.

For thinned traces, changing time units does not reconstruct the unthinned chain's integrated time: omitted lags remain unknown. Rust and notebook results can differ at roundoff, especially when a pair sum or the final time is near zero.

In [ ]:
def scalar_acf(samples: pl.Series, max_lag: int) -> list[float]:
    """Direct biased ACF for a finite scalar series at regular intervals."""
    count = len(samples)
    if count < 2:
        msg = "At least two samples are required."
        raise ValueError(msg)
    if not 0 <= max_lag < count:
        msg = "Maximum lag must be nonnegative and less than the sample count."
        raise ValueError(msg)
    if samples.null_count() or not samples.is_finite().all():
        msg = "Samples must be finite and non-null."
        raise ValueError(msg)
    origin = float(samples[0])
    centered = samples - origin
    if not centered.is_finite().all():
        centered = samples * 0.5 - origin * 0.5
    scale = max(abs(value) for value in centered)
    if scale == 0:
        msg = "Constant trace: normalized ACF is undefined."
        raise ValueError(msg)
    # Elementwise division avoids a reciprocal overflowing for subnormal scales.
    centered = pl.Series([value / scale for value in centered])
    centered = centered - math.fsum(centered) / count
    variance_sum = math.fsum(centered * centered)
    return [1.0] + [float((centered.slice(0, count - lag) * centered.slice(lag)).sum()) / variance_sum for lag in range(1, max_lag + 1)]


def initial_monotone_time(acf: list[float]) -> tuple[float, int]:
    """Return (tau, last retained lag) from the ACF computed above."""
    pairs: list[float] = []
    previous = math.inf
    for lag in range(0, len(acf) - 1, 2):
        pair = acf[lag] + acf[lag + 1]
        if pair <= 0:
            tau = 2 * math.fsum(pairs) - 1
            if tau <= 0:
                msg = "Estimated integrated time is not positive."
                raise ValueError(msg)
            return tau, lag - 1
        previous = min(previous, pair)
        pairs.append(previous)
    msg = "No truncation pair found: increase the lag budget or collect a longer trace."
    raise ValueError(msg)

In [ ]:
analysis_discard = 0
analysis_max_lag = 2_000
analysis_rows: list[dict[str, str | int | float | None]] = []
acf_curves: list[tuple[int, str, list[float]]] = []
for chain_trace in trace.partition_by("chain_id", maintain_order=True):
    chain_id = int(chain_trace["chain_id"][0])
    production = chain_trace.slice(analysis_discard)
    count = production.height
    spacing = production["step"].diff().drop_nulls()
    for observable in ("energy", "magnetization"):
        row: dict[str, str | int | float | None] = {
            "chain_id": chain_id,
            "observable": observable,
            "samples": count,
            "tau": None,
            "window": None,
            "tau_steps": None,
            "status": "unavailable",
        }
        if count >= 2 and (spacing.n_unique() != 1 or spacing[0] <= 0):
            row["status"] = "ACF requires uniform positive step spacing."
            analysis_rows.append(row)
            continue
        try:
            acf = scalar_acf(production[observable], min(analysis_max_lag, max(0, count - 1)))
            acf_curves.append((chain_id, observable, acf))
            tau, window = initial_monotone_time(acf)
            row.update(tau=tau, window=window, tau_steps=tau * int(spacing[0]))
            row["status"] = "caution: fewer than 50 estimated times" if count < 50 * tau else "estimate; check longer runs"
        except ValueError as error:
            row["status"] = str(error)
        analysis_rows.append(row)
autocorrelation_summary = pl.DataFrame(analysis_rows)
autocorrelation_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, observable in zip(axes, ("energy", "magnetization"), strict=True):
    for chain_id, name, acf in acf_curves:
        if name == observable:
            ax.plot(range(len(acf)), acf, label=f"chain {chain_id}")
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_title(f"{observable.capitalize()} ACF")
    ax.set_xlabel("lag (recorded-sample intervals)")
    ax.set_ylabel("autocorrelation")
    if ax.lines[:-1]:
        ax.legend()
fig.tight_layout()

## Multiple chains

The `chain_id` column generalizes this table to several chains. In Rust, give each chain a distinct `ChainId`, accumulate it with its own `TraceRecorder`, then merge the per-chain traces with `Trace::extend` before analysis. Keep each original chain separate for ESS and pass borrowed per-chain slices to `SplitRhat::estimate`. Distinct IDs alone do not establish independent streams or comparable targets.

## ESS, wall-clock efficiency, and split R-hat

ESS for each scalar mean is N/tau, using the retained sample count and tau in recorded-sample intervals. Anticorrelation can make ESS larger than N; this is not rank-normalized bulk or tail ESS. A small ESS means poor precision, and a large estimate from a short or nonstationary trace can be misleading.

The example now runs four sequential chains with seeds 42–45, uniform up/down and alternating starts, 5,000 warmup steps discarded per chain, and 20,000 production draws each. It exports `target/ising_1d_diagnostics.json` with per-chain durations, ESS, ESS/second, split R-hat, estimator names, warmup policy, and sample counts. Production timing includes transitions and observation/recording, but excludes warmup, export, and analysis. These measured rates vary with hardware and build profile and are not benchmark evidence. They are per-chain rates, not parallel workload throughput.

The notebook uses that companion JSON by default for repository-generated input. For external CSVs, set `MCMC_DIAGNOSTICS_PATH` explicitly to the corresponding JSON from the same run; without it, rates remain unavailable. Never reuse timings from a different run. Counts, observable names, and integrated times are checked against this analysis. If `analysis_discard` is nonzero, rates stay unavailable because the retained subset was not timed separately.

Classical split R-hat compares between-half and within-half variances after splitting every original chain. It requires at least two comparable original chains of equal length, with at least four draws each. For odd lengths it omits each middle draw. Any constant half or unresolved variance is unavailable. A value near one is not proof of convergence: this raw-moment diagnostic assumes finite marginal variance and can miss scale/tail differences. It is not the modern rank-normalized, folded diagnostic; see the [Stan definitions and limitations](https://mc-stan.org/docs/2_29/reference-manual/notation-for-samples-chains-and-draws.html). Compare longer runs, dispersed starts, ESS, and trace plots.


In [ ]:
configured_diagnostics = os.environ.get("MCMC_DIAGNOSTICS_PATH")
diagnostics_path = Path(configured_diagnostics).expanduser() if configured_diagnostics else trace_path.with_name("ising_1d_diagnostics.json")
timing_report = None
if configured_diagnostics or (not configured_trace and diagnostics_path.is_file()):
    timing_report = json.loads(diagnostics_path.read_text(encoding="utf-8"))
    if timing_report["schema_version"] != 1:
        msg = "Unsupported diagnostics schema."
        raise ValueError(msg)

elapsed_by_chain: dict[int, float] = {}
if timing_report is not None and analysis_discard == 0:
    chain_metadata = timing_report["chains"]
    expected_ids = set(trace["chain_id"].unique().to_list())
    if len(chain_metadata) != len(expected_ids) or {item["chain_id"] for item in chain_metadata} != expected_ids:
        msg = "Timing metadata must describe exactly the analyzed chains."
        raise ValueError(msg)
    for item in chain_metadata:
        chain_rows = [row for row in analysis_rows if row["chain_id"] == item["chain_id"]]
        if any(item["samples"] != row["samples"] for row in chain_rows):
            msg = "Timing sample counts do not match the analyzed trace."
            raise ValueError(msg)
        reported = {value["observable"]: value for value in item["observables"]}
        for row in chain_rows:
            tau = row["tau"]
            if isinstance(tau, float) and (
                row["observable"] not in reported
                or not isinstance(reported[row["observable"]]["tau"], (int, float))
                or not math.isclose(tau, reported[row["observable"]]["tau"], rel_tol=1e-10, abs_tol=1e-12)
            ):
                msg = "Timing diagnostics do not match this observable analysis."
                raise ValueError(msg)
        elapsed = item["elapsed_seconds"]
        if isinstance(elapsed, bool) or not isinstance(elapsed, (int, float)) or not math.isfinite(elapsed) or elapsed < 0:
            msg = "Elapsed seconds must be a finite nonnegative number."
            raise ValueError(msg)
        if elapsed > 0:
            elapsed_by_chain[item["chain_id"]] = float(elapsed)

ess_rows: list[dict[str, str | int | float | None]] = []
for row in analysis_rows:
    tau, count, chain_id = row["tau"], row["samples"], row["chain_id"]
    ess = count / tau if isinstance(count, int) and isinstance(tau, float) else None
    elapsed = elapsed_by_chain.get(chain_id) if isinstance(chain_id, int) else None
    ess_rows.append(
        {
            **row,
            "method": "single_chain_mean_geyer_initial_monotone",
            "discarded_input_rows": analysis_discard,
            "ess": ess,
            "elapsed_seconds": elapsed,
            "ess_per_second": ess / elapsed if ess is not None and elapsed is not None else None,
            "timing_status": "measured production" if elapsed is not None else "unavailable for analyzed samples",
        }
    )
ess_summary = pl.DataFrame(ess_rows)
ess_summary.write_csv(figure_output_dir / "ising_ess_summary.csv")
ess_summary

In [ ]:
def classical_split_rhat(chains: list[list[float]]) -> float:
    """Raw-moment split R-hat; no rank normalization or implicit trimming."""
    if len(chains) < 2:
        msg = "At least two original chains are required."
        raise ValueError(msg)
    count = len(chains[0])
    if any(len(chain) < 4 for chain in chains):
        msg = "At least four samples per original chain are required."
        raise ValueError(msg)
    if any(len(chain) != count for chain in chains):
        msg = "Original chains must have equal lengths."
        raise ValueError(msg)
    if any(not math.isfinite(value) for chain in chains for value in chain):
        msg = "Every sample must be finite."
        raise ValueError(msg)
    half_count = count // 2
    halves = [half for chain in chains for half in (chain[:half_count], chain[-half_count:])]
    if any(min(half) == max(half) for half in halves):
        msg = "A split chain is constant."
        raise ValueError(msg)
    origin = halves[0][0]
    low = min(min(half) for half in halves)
    high = max(max(half) for half in halves)
    factor = 0.5 if not math.isfinite(high - low) else 1.0
    scale = max(abs(value * factor - origin * factor) for half in halves for value in half)
    means, variances = [], []
    for half in halves:
        anchor = half[0]
        delta = [(value * factor - anchor * factor) / scale for value in half]
        mean_delta = math.fsum(delta) / half_count
        variance = math.fsum((value - mean_delta) ** 2 for value in delta) / (half_count - 1)
        if variance <= 0 or not math.isfinite(variance):
            msg = "Split R-hat variance is numerically unresolved."
            raise ValueError(msg)
        means.append((anchor * factor - origin * factor) / scale + mean_delta)
        variances.append(variance)
    mean = math.fsum(means) / len(means)
    between_over_n = math.fsum((value - mean) ** 2 for value in means) / (len(means) - 1)
    within = math.fsum(variances) / len(variances)
    result = math.sqrt((half_count - 1) / half_count * within + between_over_n) / math.sqrt(within)
    if not math.isfinite(result):
        msg = "Split R-hat is not representable."
        raise ValueError(msg)
    return result


def validate_rhat_spacing(chains: list[pl.DataFrame]) -> None:
    """Require the same fixed recording interval in all original chains."""
    spacings = [chain["step"].diff().drop_nulls() for chain in chains]
    if any(spacing.is_empty() or spacing.n_unique() != 1 or spacing[0] <= 0 for spacing in spacings) or len({spacing[0] for spacing in spacings}) != 1:
        msg = "R-hat requires matching uniform positive sample spacing."
        raise ValueError(msg)

In [ ]:
production_chains = [chain.slice(analysis_discard) for chain in trace.partition_by("chain_id", maintain_order=True)]
chain_ids = trace["chain_id"].unique(maintain_order=True).to_list()
sample_count = production_chains[0].height
common_samples = sample_count if all(chain.height == sample_count for chain in production_chains) else None
rhat_rows: list[dict[str, str | int | float | None]] = []
for observable in ("energy", "magnetization"):
    rhat_row: dict[str, str | int | float | None] = {
        "observable": observable,
        "method": "classical_split_rhat",
        "chain_ids": ";".join(str(value) for value in chain_ids),
        "chain_count": len(chain_ids),
        "samples_per_chain": common_samples,
        "samples_per_split_chain": None,
        "omitted_middle_draws_per_chain": None,
        "discarded_input_rows": analysis_discard,
        "rhat": None,
        "status": "unavailable",
    }
    try:
        validate_rhat_spacing(production_chains)
        rhat_row["rhat"] = classical_split_rhat([chain[observable].to_list() for chain in production_chains])
        rhat_row["samples_per_split_chain"] = sample_count // 2
        rhat_row["omitted_middle_draws_per_chain"] = sample_count % 2
        rhat_row["status"] = "estimate; inspect ESS and longer runs"
    except ValueError as error:
        rhat_row["status"] = str(error)
    rhat_rows.append(rhat_row)
rhat_summary = pl.DataFrame(rhat_rows)
rhat_summary.write_csv(figure_output_dir / "ising_rhat_summary.csv")
rhat_summary